# 02 - Hiperparámetros, Elección de K y Fronteras de Decisión en KNN

## Curso de Machine Learning: Modelos de Clasificación

En este cuaderno profundizaremos en el ajuste del hiperparámetro fundamental del algoritmo KNN: el número de vecinos $K$. Analizaremos matemáticamente el dilema **Sesgo-Varianza (Bias-Variance Tradeoff)**, compararemos esquemas de votación uniforme vs. ponderada por distancia, visualizaremos las fronteras de decisión generadas en espacios bidimensionales y aplicaremos validación cruzada sistemática con `GridSearchCV`.

---

### Contenidos
1. El Dilema Sesgo-Varianza en Función de $K$
2. Ponderación de Vecinos: Uniforme (`uniform`) vs. Ponderada (`distance`)
3. Visualización 2D de Fronteras de Decisión con `DecisionBoundaryDisplay`
4. Optimización de $K$ con Validación Cruzada (`GridSearchCV`)
5. Curva de Validación: Diagnóstico de Sobreajuste y Subajuste
6. La Maldición de la Dimensionalidad (Curse of Dimensionality)
7. Conclusiones


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_moons, load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, validation_curve
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import DecisionBoundaryDisplay

# Configuracion visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. El Dilema Sesgo-Varianza en Función de $K$

El hiperparámetro $K$ actúa como el principal regulador de la complejidad del modelo en KNN:

- **$K = 1$ (Varianza Muy Alta / Sesgo Bajo)**:
  - Cada observación se clasifica exactamente de acuerdo con su vecino más inmediato (teselación de Voronoi).
  - El error en entrenamiento es exactamente cero ($E_{train} = 0$).
  - La frontera de decisión es sumamente tortuosa y sensible a cualquier outlier o muestra ruidosa (**Sobreajuste / Overfitting**).

- **$K \approx N$ (Sesgo Muy Alto / Varianza Baja)**:
  - La predicción se convierte en la clase mayoritaria de todo el conjunto de entrenamiento para cualquier consulta.
  - La frontera se simplifica en exceso o desaparece (**Subajuste / Underfitting**).

- **$K$ Óptimo**:
  - Un valor intermedio que suaviza el ruido local pero retiene la verdadera estructura geométrica de las clases.


In [ ]:
# Generacion de dataset no lineal sintetico (make_moons) con ruido
X_moons, y_moons = make_moons(n_samples=300, noise=0.30, random_state=42)

# Visualizacion de Fronteras de Decision para diferentes valores de K
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

k_values = [1, 5, 15, 60]

for idx, k in enumerate(k_values):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_moons, y_moons)
    
    DecisionBoundaryDisplay.from_estimator(
        knn, X_moons, 
        cmap='coolwarm', 
        alpha=0.6, 
        ax=axes[idx], 
        response_method='predict'
    )
    
    scatter = axes[idx].scatter(
        X_moons[:, 0], X_moons[:, 1], 
        c=y_moons, cmap='coolwarm', edgecolor='k', s=45
    )
    
    acc_train = knn.score(X_moons, y_moons)
    axes[idx].set_title(f'K = {k} (Acc Train: {acc_train*100:.1f}%)', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Caracteristica 1')
    axes[idx].set_ylabel('Caracteristica 2')

plt.suptitle('Evolucion de la Frontera de Decision en KNN: De Alta Varianza (K=1) a Alto Sesgo (K=60)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 2. Ponderación de Vecinos: Uniforme (`uniform`) vs. Ponderada (`distance`)

Por defecto, Scikit-Learn utiliza `weights='uniform'`, donde cada uno de los $K$ vecinos tiene un voto de igual magnitud ($1$).

Con `weights='distance'`, el voto de cada vecino $i$ se pondera por el inverso de su distancia euclidiana:
$$w_i = \frac{1}{d(\mathbf{x}^*, \mathbf{x}_i)}$$

### Ventajas de `weights='distance'`:
- Vecinos extremadamente cercanos dominan la predicción, reduciendo el impacto negativo de vecinos lejanos cuando $K$ es moderadamente grande.
- Genera transiciones de probabilidad mucho más suaves y adaptativas.


In [ ]:
# Comparacion directa: weights='uniform' vs weights='distance'
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

knn_uniform = KNeighborsClassifier(n_neighbors=15, weights='uniform').fit(X_moons, y_moons)
knn_distance = KNeighborsClassifier(n_neighbors=15, weights='distance').fit(X_moons, y_moons)

DecisionBoundaryDisplay.from_estimator(
    knn_uniform, X_moons, cmap='coolwarm', alpha=0.6, ax=axes[0], response_method='predict'
)
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='coolwarm', edgecolor='k', s=45)
axes[0].set_title("K=15 con Votacion Uniforme ('uniform')", fontsize=12, fontweight='bold')

DecisionBoundaryDisplay.from_estimator(
    knn_distance, X_moons, cmap='coolwarm', alpha=0.6, ax=axes[1], response_method='predict'
)
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='coolwarm', edgecolor='k', s=45)
axes[1].set_title("K=15 con Votacion Ponderada por Distancia ('distance')", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()


## 3. Optimización de $K$ con Validación Cruzada y Curva de Validación

Para encontrar el valor de $K$ que maximice la capacidad de generalización en datos no vistos, empleamos Validación Cruzada (K-Fold CV) combinada con `validation_curve`.


In [ ]:
# Carga de dataset real y preparacion de pipeline
cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

pipeline_cv = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

param_range = np.arange(1, 41)

train_scores, val_scores = validation_curve(
    pipeline_cv, X_train, y_train,
    param_name='knn__n_neighbors',
    param_range=param_range,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

# Grafico de la Curva de Validacion
plt.figure(figsize=(11, 5.5))
plt.plot(param_range, train_mean, 'o-', color='#2b5c8f', label='Desempeño en Entrenamiento')
plt.fill_between(param_range, train_mean - train_std, train_mean + train_std, alpha=0.15, color='#2b5c8f')

plt.plot(param_range, val_mean, 's-', color='#d95f02', label='Validacion Cruzada (5-Fold CV)')
plt.fill_between(param_range, val_mean - val_std, val_mean + val_std, alpha=0.15, color='#d95f02')

best_k = param_range[np.argmax(val_mean)]
best_acc = np.max(val_mean)

plt.axvline(best_k, color='green', linestyle='--', label=f'K Optimo = {best_k} (CV Acc: {best_acc*100:.2f}%)')
plt.xlabel('Numero de Vecinos (K)', fontsize=11, fontweight='bold')
plt.ylabel('Exactitud (Accuracy)', fontsize=11, fontweight='bold')
plt.title('Curva de Validacion para el Hiperparametro K en Breast Cancer', fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


## 4. Búsqueda por Rejilla Exhaustiva (`GridSearchCV`)

Combinamos simultáneamente el número de vecinos $K$, el esquema de pesos (`uniform` vs. `distance`) y la métrica de distancia (Manhattan con $p=1$ vs. Euclidiana con $p=2$).


In [ ]:
param_grid = {
    'knn__n_neighbors': [3, 5, 7, 9, 11, 13, 15, 21],
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2] # 1: Manhattan, 2: Euclidiana
}

grid_search = GridSearchCV(
    pipeline_cv,
    param_grid=param_grid,
    cv=5,
    scoring='f1', # Optimizamos F1-Score para considerar precision y recall
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(f"Mejores Hiperparametros encontrados: {grid_search.best_params_}")
print(f"Mejor F1-Score en Validación Cruzada: {grid_search.best_score_:.4f}")

# Evaluacion en conjunto de prueba independiente
mejor_modelo = grid_search.best_estimator_
test_acc = mejor_modelo.score(X_test, y_test)
print(f"Exactitud Final en Conjunto de Prueba: {test_acc * 100:.2f}%")


## 5. La Maldición de la Dimensionalidad (Curse of Dimensionality)

Un límite teórico crucial de KNN ocurre a medida que el número de características $D$ crece significativamente ($D > 50$):
1. **El espacio se vuelve hiperdisperso**: La distancia promedio entre dos puntos arbitrarios converge a un valor constante, haciendo que todos los vecinos parezcan equidistantes.
2. **Degradación computacional**: Las estructuras de árbol indexadas (`KDTree` y `BallTree`) degeneran en búsqueda exhaustiva de fuerza bruta $O(N \cdot D)$.
3. **Remedio práctico**: Aplicar técnicas previas de selección o reducción de dimensionalidad como **PCA (Principal Component Analysis)**.


## 6. Conclusiones

1. **Equilibrio Sesgo-Varianza**: Valores pequeños de $K$ conllevan sobreajuste; valores excesivamente altos conllevan subajuste.
2. **Pesos por distancia**: Mitigan la influencia de observaciones distantes cuando se usan valores mayores de $K$.
3. **Validación Cruzada**: Permite determinar objetivamente la configuración ideal de vecinos y norma geométrica.
4. **Dimensionalidad**: En datasets masivos con cientos o miles de variables, es esencial realizar reducción de dimensionalidad previa a KNN.
